In [2]:
# Project 8: Food Delivery Operations Analytics

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 109. Load and inspect dataset
df = pd.read_csv("08_food_delivery.csv")

print("\n109. DATASET INSPECTION")
print(df.head())
print(df.shape)
print(df.info())

# 110. Find and handle missing delivery times
print("\n110. MISSING VALUES")
print(df["Delivery_Time_Minutes"].isnull().sum())

for city in df["City"].unique():
    median_time = df.loc[
        df["City"] == city,
        "Delivery_Time_Minutes"
    ].median()

    df.loc[
        (df["City"] == city) &
        (df["Delivery_Time_Minutes"].isnull()),
        "Delivery_Time_Minutes"
    ] = median_time

print("Missing values after filling:")
print(df["Delivery_Time_Minutes"].isnull().sum())

# 111. Check duplicate Order_ID
print("\n111. DUPLICATE ORDERS")
print(df.duplicated(subset="Order_ID").sum())

df = df.drop_duplicates(
    subset="Order_ID",
    keep="first"
)

print("Total rows after removing duplicates:", len(df))

df["Delivery_Time_Minutes"] = (
    df["Delivery_Time_Minutes"].astype(int)
)

# Convert order date and extract hour
df["Order_Date"] = pd.to_datetime(df["Order_Date"])

df["Order_Hour"] = pd.to_datetime(
    df["Order_Time"],
    format="%H:%M"
).dt.hour

# 112. Total orders and status percentages
print("\n112. ORDER STATUS")
print("Total orders:", len(df))

status_count = df["Order_Status"].value_counts()
status_percent = (
    df["Order_Status"].value_counts(normalize=True) * 100
)

print(status_count)
print(status_percent.round(2))

order_status_summary = pd.DataFrame({
    "Count": status_count,
    "Percentage": status_percent
})
order_status_summary.to_csv("order_status_summary.csv")

# 114. Top five restaurants by total order value
print("\n114. TOP FIVE RESTAURANTS")
restaurant_value = df.groupby("Restaurant")[
    "Order_Value"
].sum()

top_five = restaurant_value.sort_values(
    ascending=False
).head(5)
print(top_five)
top_five.to_csv("top_five_restaurants.csv")

# 115. Average order value by restaurant
print("\n115. AVERAGE ORDER VALUE")
average_value = df.groupby("Restaurant")[
    "Order_Value"
].mean().round(2)
print(average_value)
average_value.to_csv("average_order_value_by_restaurant.csv")

# 116. Peak ordering hours
print("\n116. PEAK ORDERING HOURS")
peak_hours = df.groupby("Order_Hour").size()
print(peak_hours.sort_values(ascending=False))
peak_hours.to_csv("orders_by_hour.csv", header=["Order_Count"])

# 117. Delivery time by order status
print("\n117. DELIVERY TIME BY STATUS")
status_delivery = df.groupby("Order_Status")[
    "Delivery_Time_Minutes"
].mean().round(2)
print(status_delivery)
status_delivery.to_csv("delivery_time_by_status.csv")

# 118. Cancellation and delay rates by city
print("\n118. CITY STATUS PERCENTAGES")
city_status = pd.crosstab(
    df["City"],
    df["Order_Status"],
    normalize="index"
) * 100
print(city_status.round(2))
city_status.to_csv("cancellation_delay_rates_by_city.csv")

# 119. Median and standard deviation using NumPy
print("\n119. DELIVERY STATISTICS")
delivery = df["Delivery_Time_Minutes"].to_numpy()
median_time = np.median(delivery)
standard_deviation = np.std(delivery)
print("Median:", median_time)
print("Standard deviation:", standard_deviation)

delivery_stats = pd.DataFrame({
    "Metric": ["Median", "Standard Deviation"],
    "Value": [median_time, standard_deviation]
})
delivery_stats.to_csv("delivery_time_statistics.csv", index=False)

# 120. Unusually long deliveries using IQR
print("\n120. IQR ANALYSIS")
q1 = np.percentile(delivery, 25)
q3 = np.percentile(delivery, 75)
iqr = q3 - q1
upper_limit = q3 + (1.5 * iqr)
print("Q1:", q1)
print("Q3:", q3)
print("IQR:", iqr)
print("Upper limit:", upper_limit)

long_deliveries = df[
    df["Delivery_Time_Minutes"] > upper_limit
]
print("Long deliveries:")
print(long_deliveries.head())
long_deliveries.to_csv("unusually_long_deliveries.csv", index=False)

# 121. Create delivery time categories
print("\n121. DELIVERY TIME BANDS")
def delivery_band(time):
    if time <= 30:
        return "Fast"
    elif time <= 45:
        return "Standard"
    else:
        return "Delayed"

df["Delivery_Time_Band"] = df[
    "Delivery_Time_Minutes"
].apply(delivery_band)

delivery_bands_count = df["Delivery_Time_Band"].value_counts()
print(delivery_bands_count)
delivery_bands_count.to_csv("delivery_time_bands.csv")

# 122. Revenue by city and payment mode
print("\n122. REVENUE ANALYSIS")
revenue = df.pivot_table(
    values="Order_Value",
    index="City",
    columns="Payment_Mode",
    aggfunc="sum",
    fill_value=0
)
print(revenue)
revenue.to_csv("revenue_by_city_payment_mode.csv")

# City performance table
city_performance = df.groupby("City").agg(
    Total_Orders=("Order_ID", "count"),
    Average_Delivery=("Delivery_Time_Minutes", "mean"),
    Total_Revenue=("Order_Value", "sum")
)
print(city_performance)
city_performance.to_csv("city_performance_table.csv")

# 123. Five operational recommendations
print("\n123. RECOMMENDATIONS")
print("1. Improve delivery services in cities with longer delivery times.")
print("2. Arrange more delivery staff during peak hours.")
print("3. Monitor restaurants with more delayed orders.")
print("4. Reduce cancellation rates by improving order handling.")
print("5. Investigate unusually long delivery times.")

# Save CSV output files
df.to_csv("cleaned_food_delivery.csv", index=False)

print("\nPROJECT COMPLETED SUCCESSFULLY")


109. DATASET INSPECTION
  Order_ID  Order_Date Order_Time    City      Restaurant Order_Status  \
0  ORD0001  2026-02-25      13:00  Mumbai  Tandoori Treat    Delivered   
1  ORD0002  2026-03-02      17:45  Jaipur     Royal Thali    Delivered   
2  ORD0003  2026-02-15      21:30   Delhi     Pizza Point    Delivered   
3  ORD0004  2026-02-07      23:10   Delhi     Curry House    Delivered   
4  ORD0005  2026-02-16      20:50   Delhi     Curry House      Delayed   

   Delivery_Time_Minutes  Order_Value Payment_Mode  
0                   35.0       325.38         Cash  
1                   46.0       279.55       Wallet  
2                   12.0       513.69       Wallet  
3                   18.0       808.51          UPI  
4                   41.0       868.14         Cash  
(303, 9)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 303 entries, 0 to 302
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  ---